# av-ecs-poc – test the deployed servers

Runs the README test scenarios against the deployed POC and shows the results as tables:

| # | Scenario | What it proves |
|---|---|---|
| 1 | Addresses | Every server and guard task registered its own task IP; `hostname -i` matches the metadata IP |
| 2 | Registration | Devices have an owner server and a specific Guard endpoint (`GUARD_EXTERNAL_IP:port`) |
| 3 | Sticky path | Director → owning server → that Guard → agent; the same path on every round |
| 4 | Long operation | A 300 s+ command survives every hop (optional, about 5 minutes) |
| 5 | Server replaced | A stopped server's devices are rehomed; commands keep working |
| 6 | Guard replaced | Tunnels move to another Guard; new endpoints are registered |
| 7 | Rolling deploy | Every server is replaced; all devices end up on new servers (optional) |

**Requirements**
- The notebook must reach the Director NLB on port 11000, so it must be inside `ClientCidr`.
- The notebook role needs: `cloudformation:DescribeStacks`, `ecs:ListTasks`, `ecs:DescribeTasks`,
  `ecs:StopTask`, `ecs:UpdateService` and `ecs:DescribeServices` (scenarios 5–7), and
  `logs:StartQuery`/`logs:GetQueryResults` (scenario 1).

## Settings

In [ ]:
REGION      = "us-east-1"
PROJECT     = "av-ecs-poc"
SVC_STACK   = f"{PROJECT}-services"
CLUSTER     = PROJECT
DIRECTOR_URL = None          # leave None to read it from the services stack outputs

ROUNDS          = 3          # sweep rounds in scenario 3
RUN_LONG_OP     = False      # scenario 4 takes ~5 minutes
LONG_OP_SECONDS = 310
RUN_ROLLING     = False      # scenario 7 replaces every server task

## Helpers

In [ ]:
import json, time, urllib.error, urllib.request
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 250)

def call(method, path, body=None, timeout=400):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(DIRECTOR_URL.rstrip("/") + path, data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.load(r)
    except urllib.error.HTTPError as e:
        try:
            return e.code, json.load(e)
        except Exception:
            return e.code, {"error": str(e)}

def path(trace):
    """director[task@ip] -> server[task@ip] -> guard[task@ip:port] -> agent[...]"""
    return " -> ".join(f"{h['hop']}[{h.get('task','')[:8]}@{h.get('endpoint') or h.get('ip')}]"
                       + ("(fwd)" if h.get("note") else "") for h in (trace or []))

def servers():
    df = pd.DataFrame(call("GET", "/servers")[1]["servers"])
    if not df.empty:
        df["server"] = df["server_id"].str[:12]
        df["address"] = df["address"] + ":" + df["port"].astype(str)
        df = df[["server", "address", "az", "live", "devices", "started_at", "last_heartbeat"]]
    return df

def devices():
    df = pd.DataFrame(call("GET", "/devices")[1]["devices"])
    if not df.empty:
        for c in ("owner_server_id", "guard_id", "registered_via"):
            df[c] = df[c].fillna("").str[:12]
        df = df[["device_id", "online", "owner_server_id", "owner_address", "guard_id", "guard_endpoint",
                 "registered_via", "rehome_count"]]
    return df

def command(device, cmd="status", delay=0):
    t0 = time.time()
    code, res = call("POST", f"/devices/{device}/command", {"cmd": cmd, "delay": delay})
    return {"device_id": device, "http": code, "seconds": round(time.time() - t0, 1),
            "result": res.get("result") or res.get("error"), "path": path(res.get("trace"))}

def sweep(rounds=1, pause=3):
    """Command every device each round; flag any device whose path changed."""
    rows, last = [], {}
    for r in range(1, rounds + 1):
        for dev in devices()["device_id"]:
            row = {"round": r, **command(dev)}
            row["changed"] = dev in last and last[dev] != row["path"]
            last[dev] = row["path"]
            rows.append(row)
        if r < rounds:
            time.sleep(pause)
    return pd.DataFrame(rows)

def wait_for(check, timeout=180, every=5, label="condition"):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if check():
            print(f"{label}: met after {time.time() - t0:.0f}s")
            return True
        time.sleep(every)
    print(f"{label}: NOT met within {timeout}s")
    return False

def ok(flag, message):
    print(("PASS  " if flag else "FAIL  ") + message)
    return flag

## Connect to the Director

In [ ]:
import boto3

session = boto3.Session(region_name=REGION)
ecs, logs, cfn = session.client("ecs"), session.client("logs"), session.client("cloudformation")

if not DIRECTOR_URL:
    outs = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=SVC_STACK)["Stacks"][0]["Outputs"]}
    DIRECTOR_URL = outs["DirectorUrl"]
print("Director:", DIRECTOR_URL)
print(call("GET", "/healthz", timeout=10))

## Optional: open the browser test page through this notebook

If your workstation can't reach the Director NLB directly, this cell forwards `localhost:11000` on the notebook
instance to the NLB. Open the page through Jupyter's proxy: replace the end of this notebook's URL with
`/proxy/11000/`, keeping the trailing slash. On a SageMaker notebook instance that's
`https://<instance>.notebook.<region>.sagemaker.aws/proxy/11000/`.
The forwarder runs until you restart the kernel.

In [ ]:
import socket, threading
from urllib.parse import urlparse

def start_forwarder(local_port, target_host, target_port):
    def pipe(a, b):
        try:
            while (data := a.recv(65536)):
                b.sendall(data)
        except OSError:
            pass
        finally:
            for s in (a, b):
                try: s.close()
                except OSError: pass
    def handle(client):
        try:
            upstream = socket.create_connection((target_host, target_port), timeout=10)
        except OSError as exc:
            print("forwarder: cannot reach the Director NLB:", exc)
            client.close()
            return
        upstream.settimeout(None)
        threading.Thread(target=pipe, args=(client, upstream), daemon=True).start()
        pipe(upstream, client)
    def serve():
        srv = socket.socket(); srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(("127.0.0.1", local_port)); srv.listen(50)
        while True:
            client, _ = srv.accept()
            threading.Thread(target=handle, args=(client,), daemon=True).start()
    threading.Thread(target=serve, daemon=True).start()

u = urlparse(DIRECTOR_URL)
start_forwarder(11000, u.hostname, u.port or 11000)
print(f"Forwarding localhost:11000 -> {u.hostname}:{u.port}. Open <this notebook's base URL>/proxy/11000/")

## 1. Addresses

Each server registered `NG_SERVER_NAME:8261`, set to its task IP by the wrapper entrypoint. The `address_resolved` lines show what each source returned (`hostname -i`, `getent hosts`, task metadata), which one the wrapper used (`source`), and whether it equals the task's metadata IP (`match`).

In [ ]:
s = servers()
display(s)
ok(len(s[s["live"]]) > 0, f"{len(s[s['live']])} live server(s) registered with their task IP")
ok(s["address"].is_unique, "every server address is unique")

In [ ]:
q = logs.start_query(
    logGroupNames=[f"/{PROJECT}/server", f"/{PROJECT}/guard"],
    startTime=int(time.time()) - 6 * 3600, endTime=int(time.time()),
    queryString=('fields @timestamp, role, hostname, hostname_i, getent_ip, metadata_ip, address, source, match '
                 '| filter event = "address_resolved" | sort @timestamp desc | limit 50'))
while (res := logs.get_query_results(queryId=q["queryId"]))["status"] in ("Running", "Scheduled"):
    time.sleep(1)
addr = pd.DataFrame([{f["field"]: f["value"] for f in r if not f["field"].startswith("@ptr")} for r in res["results"]])
display(addr)
if not addr.empty:
    ok((addr["match"] == "yes").all(), "the wrapper exported the task metadata IP on every task")
    ok((addr["source"] == "hostname -i").all(), "hostname -i alone was enough on every task")

## 2. Registration

Every device has an owner server and a specific Guard endpoint. If `registered_via` differs from `owner_server_id`, the Guard's call landed on another server through Service Connect, which forwarded it to the owner (server → server).

In [ ]:
d = devices()
display(d)
ok(len(d) > 0 and d["online"].all(), f"{len(d)} devices registered and online")
ok(d["owner_server_id"].nunique() > 1, f"devices spread over {d['owner_server_id'].nunique()} servers")
ok((d["registered_via"] != d["owner_server_id"]).any(), "at least one registration was forwarded server -> server")

## 3. Sticky command path

Each device should take the same path every round: Director → owning server (task IP) → the Guard holding its tunnel (task IP and device port) → agent.

In [ ]:
first = command(d["device_id"].iloc[0])
print(first["path"])
res3 = sweep(rounds=ROUNDS)
display(res3)
ok((res3["http"] == 200).all(), "every command succeeded")
ok(not res3["changed"].any(), "no device changed path between rounds (sticky)")

## 4. Long operation (optional)

A command that takes `LONG_OP_SECONDS` end to end. It tests the NLB idle timeout, the direct Director → server and server → Guard connections, and the agent tunnel.

In [ ]:
if RUN_LONG_OP:
    dev = d["device_id"].iloc[1 % len(d)]
    print(f"Running a {LONG_OP_SECONDS}s operation on {dev} ...")
    r4 = command(dev, cmd="dispense", delay=LONG_OP_SECONDS)
    display(pd.DataFrame([r4]))
    ok(r4["http"] == 200 and r4["seconds"] >= LONG_OP_SECONDS, f"long operation completed in {r4['seconds']}s")
else:
    print("Skipped - set RUN_LONG_OP = True")

## 5. Server replaced (new task, new IP)

Stops the server task that owns the most devices. After the heartbeat goes stale (30 s), its devices are rehomed to live servers, and ECS starts a replacement with a new IP.

In [ ]:
before = devices()
victim = before["owner_server_id"].value_counts().idxmax()
victim_devices = set(before.loc[before["owner_server_id"] == victim, "device_id"])
arn = next(t for t in ecs.list_tasks(cluster=CLUSTER, serviceName=f"{PROJECT}-server")["taskArns"] if t.endswith(victim) or victim in t)
ecs.stop_task(cluster=CLUSTER, task=arn, reason="POC test - server replaced")
print(f"Stopped server {victim} owning {len(victim_devices)} devices: {sorted(victim_devices)}")

wait_for(lambda: not devices()["owner_server_id"].isin([victim]).any(), timeout=180, label="devices rehomed")
after = devices()
display(after[after["device_id"].isin(victim_devices)])
display(servers())
res5 = sweep(rounds=1)
display(res5)
ok((res5["http"] == 200).all(), "all commands work after the server was replaced")

## 6. Guard replaced

Stops one Guard task. Its agents' tunnels drop and reconnect through the NLB to another Guard, which registers new `GUARD_EXTERNAL_IP:port` endpoints.

In [ ]:
before = devices()
guard_tasks = ecs.list_tasks(cluster=CLUSTER, serviceName=f"{PROJECT}-guard")["taskArns"]
victim_guard = before["guard_id"].value_counts().idxmax()
arn = next(t for t in guard_tasks if victim_guard in t)
moved = set(before.loc[before["guard_id"] == victim_guard, "device_id"])
ecs.stop_task(cluster=CLUSTER, task=arn, reason="POC test - guard replaced")
print(f"Stopped guard {victim_guard} holding {len(moved)} tunnels")

def tunnels_moved():
    now = devices()
    sub = now[now["device_id"].isin(moved)]
    return len(sub) == len(moved) and sub["online"].all() and not (sub["guard_id"] == victim_guard).any()

wait_for(tunnels_moved, timeout=180, label="tunnels reconnected to another guard")
after = devices()
cmp = before[["device_id", "guard_endpoint"]].merge(after[["device_id", "guard_endpoint"]], on="device_id",
                                                    suffixes=("_before", "_after"))
display(cmp[cmp["device_id"].isin(moved)])
res6 = sweep(rounds=1)
display(res6)
ok((res6["http"] == 200).all(), "all commands work after the guard was replaced")

## 7. Rolling deploy of the servers (optional)

Replaces every server task. All devices should end up owned by the new servers.

In [ ]:
if RUN_ROLLING:
    old_ids = set(servers().query("live")["server"])
    ecs.update_service(cluster=CLUSTER, service=f"{PROJECT}-server", forceNewDeployment=True)
    print("Rolling deployment started - waiting for the service to stabilise (several minutes)")
    ecs.get_waiter("services_stable").wait(cluster=CLUSTER, services=[f"{PROJECT}-server"])
    wait_for(lambda: not devices()["owner_server_id"].isin(old_ids).any(), timeout=180, label="all devices on new servers")
    display(servers())
    res7 = sweep(rounds=1)
    display(res7)
    ok((res7["http"] == 200).all(), "all commands work after the rolling deploy")
else:
    print("Skipped - set RUN_ROLLING = True")

## Final state

In [ ]:
display(servers())
display(devices())